# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step demonstration for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This dataset contains protein-level data extracted from mass spectrometry experiments on human mast cell-derived extracellular vesicles, with rich metadata described in a Croissant schema.

## Dataset Source
The dataset source is provided as a Croissant schema (JSON-LD file):

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We load the Croissant metadata and instantiate the dataset with `mlcroissant`. The schema describes record sets (tables), fields, columns, and file structure accessible through their `@id` fields, following Croissant best practices.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for dataset metadata
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Instantiate the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print general dataset info
print(f"Dataset: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

We explore the list of available record sets and their field IDs using the `mlcroissant` API. All entities have unique `@id` identifiers, and these are used throughout the notebook.

In [ ]:
# List all record set @id's and their fields

print('Record Sets in Dataset:')
for record_set in dataset.metadata.record_sets:
    print(f"- Record set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print('')

## 3. Data Extraction

We will extract all records from each record set using `@id`. The list of record set IDs is determined from the previous overview step. Data is loaded into Pandas DataFrames for further processing.

**Note:** Replace `<record_set_id>` with the record set `@id` you wish to inspect.

In [ ]:
# Collect record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load all records for each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading data for record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'  Columns for {record_set_id}: {df.columns.tolist()}')
    print(f'  Number of records: {len(df)}')
    print('')

# For further inspection, select the main record set (update if needed):
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate filtering, numeric normalization, and grouping operations using field IDs, as returned by the Croissant schema.

**Note:** Update `numeric_field_id` and `group_field_id` with precise `@id` values from the overview above.

In [ ]:
# Parameters: Set the numeric field @id and a categorical/group field @id
# Example values for demonstration (adjust as appropriate from section 2 above):
numeric_field_id = None  # E.g., 'cr:molecular_weight' or a similar @id in your record set
group_field_id = None    # E.g., 'cr:peptide_count' or a class/grouping variable

# If the record set has been loaded
if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records in '{main_record_set_id}' with {numeric_field_id} > {threshold}")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First 5 normalized records for '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}' (showing top 5):")
        display(grouped_df.head())
else:
    print("Please set 'numeric_field_id' and 'group_field_id' according to your dataset columns (see section 3 output).")

## 5. Visualization

We visualize the distribution of a numeric field or the relationship between two fields using Seaborn/Matplotlib. Update your feature IDs below as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Please set 'numeric_field_id' and 'group_field_id' to valid column names.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and read Croissant metadata with `mlcroissant`,
- List all available record sets and field `@id`s,
- Load data into DataFrames for each record set,
- Conduct filtering, normalization, and simple aggregation/group analysis,
- Visualize key numeric fields and groupings for exploratory insights.

For further analysis or machine learning workflows, use the field and record set `@id`s reported in step 2, which ensures reproducibility and transparency in your data extraction and pipeline design.

For more details on Croissant and `mlcroissant`, see:
- [Croissant specification](https://mlcommons.org/croissant)
- [`mlcroissant` documentation](https://mlcommons.org/croissant/docs/mlcroissant/latest/)

Happy analyzing! 🚀